# Day 7 — Reading & Writing Data

**What you will learn:**
- Reading CSV, JSON, and Parquet with options
- Defining schemas explicitly with `StructType` / `StructField`
- Write modes: `overwrite`, `append`, `ignore`, `error`
- `coalesce()` vs `repartition()` — exam favourite
- `partitionBy()` on write
- Mini project: read → clean → write

**Exam relevance:** DataFrame API (30%) — reading/writing and schema definition appear on every practice test.

## 1. Reading CSV

| Option | Default | What it does |
|---|---|---|
| `header` | `false` | First row is column names |
| `inferSchema` | `false` | Scan data to guess types (slow on large files) |
| `sep` | `,` | Column delimiter |
| `nullValue` | `""` | String to treat as null |
| `dateFormat` | `yyyy-MM-dd` | Format for date columns |
| `mode` | `PERMISSIVE` | How to handle corrupt records |

In [ ]:
# Databricks has built-in sample datasets — perfect for practice
csv_path = "/databricks-datasets/flights/departuredelays.csv"

# Read with options
flights_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(csv_path)

flights_df.printSchema()
flights_df.show(5)

## 2. Defining a Schema Explicitly (StructType)

**Why explicit schemas?**
- `inferSchema` requires a full scan of the data — slow on large files
- Inferred types can be wrong (e.g., a zip code inferred as integer)
- Production pipelines should always define schemas explicitly

> **Exam tip:** Know `StructType`, `StructField`, and the common types: `StringType`, `IntegerType`, `LongType`, `DoubleType`, `BooleanType`, `TimestampType`, `DateType`.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Define schema explicitly — faster and safer than inferSchema
schema = StructType([
    StructField("date",    StringType(),  nullable=True),
    StructField("delay",   IntegerType(), nullable=True),
    StructField("distance",IntegerType(), nullable=True),
    StructField("origin",  StringType(),  nullable=True),
    StructField("destination", StringType(), nullable=True),
])

flights_schema_df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv(csv_path)

flights_schema_df.printSchema()
flights_schema_df.show(5)

## 3. Reading JSON and Parquet

In [ ]:
# JSON — schema is inferred from JSON structure
# (Databricks has sample JSON datasets)
json_path = "/databricks-datasets/iot/iot_devices.json"

iot_df = spark.read.json(json_path)
iot_df.printSchema()
iot_df.show(3)

In [ ]:
# Parquet — columnar format, schema is embedded in the file
parquet_path = "/databricks-datasets/samples/population-vs-price/data_geo.csv"

# First let's write our flights data as parquet so we can read it back
flights_schema_df.limit(1000).write \
    .mode("overwrite") \
    .parquet("/tmp/flights_parquet")

# Now read it back — schema is embedded, no options needed
flights_from_parquet = spark.read.parquet("/tmp/flights_parquet")
flights_from_parquet.printSchema()
flights_from_parquet.show(3)

## 4. Writing Data — Modes

| Mode | Behaviour | When to use |
|---|---|---|
| `overwrite` | Delete existing data, write new | Full refresh |
| `append` | Add to existing data | Incremental load |
| `ignore` | Do nothing if path exists | Idempotent first-time writes |
| `error` / `errorifexists` | Raise error if path exists | **Default** — fail-safe |

> **Exam tip:** The default mode is `error`. If you run `.write.parquet(path)` twice without a mode, the second run fails.

In [ ]:
from pyspark.sql.functions import col

# Create a small cleaned DataFrame to write
clean_df = flights_schema_df \
    .filter(col("delay").isNotNull()) \
    .filter(col("origin").isNotNull()) \
    .limit(500)

# Write as CSV
clean_df.write.mode("overwrite").option("header", "true").csv("/tmp/clean_flights_csv")

# Write as Parquet (preferred for analytics)
clean_df.write.mode("overwrite").parquet("/tmp/clean_flights_parquet")

# Write as JSON
clean_df.write.mode("overwrite").json("/tmp/clean_flights_json")

print("Write complete")

## 5. coalesce() vs repartition()

Both change the number of partitions — but they work very differently.

| | `repartition(n)` | `coalesce(n)` |
|---|---|---|
| Can increase partitions | Yes | No |
| Can decrease partitions | Yes | Yes |
| Shuffle | **Always** — full shuffle | **No shuffle** — merges local partitions |
| Data distribution | Balanced | Potentially uneven |
| Use when | Redistributing / increasing | Reducing before write |

> **Exam tip:** Use `coalesce()` to reduce partitions before writing (no shuffle = faster).  
> Use `repartition()` when you need balanced distribution or want to increase partitions.

In [ ]:
print(f"Default partitions: {clean_df.rdd.getNumPartitions()}")

# repartition — full shuffle, balanced
df_repart = clean_df.repartition(4)
print(f"After repartition(4): {df_repart.rdd.getNumPartitions()}")

# coalesce — no shuffle, merge local
df_coalesced = clean_df.coalesce(2)
print(f"After coalesce(2): {df_coalesced.rdd.getNumPartitions()}")

# repartition by column — routes same-key rows to same partition
df_by_origin = clean_df.repartition(4, col("origin"))
print(f"After repartition by column: {df_by_origin.rdd.getNumPartitions()}")

## 6. partitionBy() on Write

Writing with `partitionBy()` creates a **directory hierarchy** based on column values.  
This enables **partition pruning** — queries that filter on the partition column skip unneeded directories entirely.

In [ ]:
# Write partitioned by origin airport
clean_df.write \
    .mode("overwrite") \
    .partitionBy("origin") \
    .parquet("/tmp/flights_partitioned")

# Spark creates: /tmp/flights_partitioned/origin=ATL/part-*.parquet
#                                          origin=ORD/part-*.parquet  etc.

# Read back — Spark automatically infers partition columns from directory names
partitioned_df = spark.read.parquet("/tmp/flights_partitioned")
partitioned_df.printSchema()  # origin column is there

# This query uses partition pruning — only reads ATL directory
partitioned_df.filter(col("origin") == "ATL").count()

## 7. Mini Project — Week 1 Capstone

Bring it all together: read raw data → apply schema → clean → transform → write Parquet.

In [ ]:
from pyspark.sql.functions import col, when, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Step 1 — Read with explicit schema
schema = StructType([
    StructField("date",        StringType(),  True),
    StructField("delay",       IntegerType(), True),
    StructField("distance",    IntegerType(), True),
    StructField("origin",      StringType(),  True),
    StructField("destination", StringType(),  True),
])

raw = spark.read.option("header", "true").schema(schema) \
           .csv("/databricks-datasets/flights/departuredelays.csv")

# Step 2 — Clean: drop nulls, filter bad data
clean = raw \
    .dropna(subset=["delay", "origin", "destination"]) \
    .filter(col("distance") > 0)

# Step 3 — Transform: add derived columns
transformed = clean \
    .withColumn("is_delayed", when(col("delay") > 0, True).otherwise(False)) \
    .withColumn("delay_category",
        when(col("delay") <= 0,   "On Time")
       .when(col("delay") <= 30,  "Minor Delay")
       .when(col("delay") <= 120, "Major Delay")
       .otherwise("Severe Delay")
    )

# Step 4 — Write as Parquet partitioned by origin
transformed.coalesce(4) \
    .write.mode("overwrite") \
    .partitionBy("origin") \
    .parquet("/tmp/week1_project_output")

print(f"Rows written: {transformed.count()}")
transformed.groupBy("delay_category").count().orderBy("delay_category").show()

---

## Day 7 Checklist

- [ ] Read a CSV with explicit `header`, `sep`, `nullValue` options
- [ ] Defined a schema with `StructType` and `StructField`
- [ ] Read JSON and Parquet
- [ ] Know all 4 write modes and their behaviour
- [ ] Know the default write mode is `error`
- [ ] Know when to use `coalesce()` vs `repartition()`
- [ ] Wrote data with `partitionBy()` and read it back
- [ ] Completed the mini project: raw → schema → clean → transform → write

---

## Week 1 Complete

You now have the full foundation:
- Spark architecture + execution model
- Lazy evaluation + Catalyst optimizer
- Column operations, filtering, sorting, null handling
- Spark SQL
- Reading and writing in all major formats

**Week 2:** Aggregations, Joins, String/Date/Array functions, UDFs, Window functions, Caching